# KrishiInsights — Agricultural Exploratory Data Analysis (EDA)

**Project:** KrishiInsights (Agricultural Data Analytics & Crop Yield Intelligence Platform)  
**Database:** Neon Serverless PostgreSQL (`neondb`)  
**Stage:** 4 (Statistical Analysis & Exploratory Data Analysis)

### Objectives
1. Inspect metadata and high-level distribution of agricultural records.
2. Calculate national descriptive statistics with strict unit segregation (Metric Tonnes vs Number of Nuts).
3. Compute historical production, area, and aggregate yield trends (1997–2015) with YoY percentage growth.
4. Analyze seasonal production distribution across Kharif, Rabi, Summer, Autumn, Winter, and Whole Year.


In [1]:
import os
import sys
sys.path.insert(0, os.path.abspath("../.."))

import pandas as pd
from ml.src.analytics.agricultural_eda import (
    calculate_overall_summary_statistics,
    get_annual_production_trends,
    get_seasonal_distribution,
    get_yearly_record_distribution
)

print("Modules successfully imported.")


Modules successfully imported.


## 1. High-Level Summary Statistics
Isolates Field Crops (`Tonnes/Hectare`) from Coconut (`Nuts/Hectare`).


In [2]:
stats = calculate_overall_summary_statistics()

print("=== Agricultural Dataset Metadata ===")
for k, v in stats["metadata"].items():
    print(f"  {k}: {v:,}" if isinstance(v, (int, float)) else f"  {k}: {v}")

print("\n=== Field Crops Summary (Metric Tonnes / Hectares) ===")
for k, v in stats["field_crops"].items():
    print(f"  {k}: {v}")

print("\n=== Coconut Summary (Number of Nuts / Hectares) ===")
for k, v in stats["coconut"].items():
    print(f"  {k}: {v}")


=== Agricultural Dataset Metadata ===
  total_records: 242,296
  num_states: 33
  num_districts: 652
  num_crops: 124
  num_seasons: 6
  min_year: 1,997
  max_year: 2,015

=== Field Crops Summary (Metric Tonnes / Hectares) ===
  crop_group: Field Crops (Tonnes)
  count: 240338
  mean_area: 12150.5
  median_area: 600.0
  std_area: 51012.27
  min_area: 0.1
  max_area: 8580100.0
  iqr_area: 4410.0
  mean_production: 45213.18
  median_production: 704.0
  std_production: 368529.83
  min_production: 0.0
  max_production: 20049700.0
  iqr_production: 6630.0
  mean_yield: 4.4187
  median_yield: 1.0
  std_yield: 12.1075
  min_yield: 0.0
  max_yield: 188.415
  iqr_yield: 1.7787

=== Coconut Summary (Number of Nuts / Hectares) ===
  crop_group: Coconut (Number of Nuts)
  count: 1958
  mean_area: 14472.25
  median_area: 1926.0
  std_area: 26972.08
  min_area: 1.0
  max_area: 149419.0
  iqr_area: 14010.25
  mean_production: 66384897.45
  median_production: 239156.5
  std_production: 177991513.67
  

## 2. Annual Production, Area, and Aggregate Yield Trends (1997–2015)
Computes mathematically sound national yield:
$$\text{Aggregate Yield} = \frac{\sum \text{Production}}{\sum \text{Area}}$$


In [3]:
trends_df = get_annual_production_trends(unit_filter="Tonnes/Hectare")
display_cols = ["year", "total_production", "total_area", "aggregate_yield", "production_yoy_pct", "yield_yoy_pct", "production_index_1997"]
print(trends_df[display_cols].to_string(index=False))


 year  total_production   total_area  aggregate_yield  production_yoy_pct  yield_yoy_pct  production_index_1997
 1997      537509212.00 230366488.00           2.3333                 NaN            NaN             100.000000
 1998      529007436.00 165343895.00           3.1994           -1.581699      37.119102              98.418301
 1999      574848715.40 157332910.00           3.6537            8.665526      14.199537             106.946765
 2000      569300219.30 163852418.00           3.4745           -0.965210      -4.904617             105.914505
 2001      584254298.47 163838553.67           3.5660            2.626747       2.633472             108.696611
 2002      546024096.14 155529040.21           3.5108           -6.543418      -1.547953             101.584137
 2003      562230567.33 167960808.80           3.3474            2.968087      -4.654210             104.599243
 2004      525477953.17 165916827.33           3.1671           -6.536929      -5.386270              97

## 3. Seasonal Agricultural Breakdown
Examines production and cultivated area shares across crop seasons.


In [4]:
seasonal_df = get_seasonal_distribution(exclude_coconut=True)
print(seasonal_df[["season", "record_count", "total_production", "total_area", "aggregate_yield", "production_share_pct", "area_share_pct"]].to_string(index=False))


    season  record_count  total_production   total_area  aggregate_yield  production_share_pct  area_share_pct
Whole Year         54125      4.115583e+09 2.259718e+08          18.2128             37.874231        7.738163
    Kharif         94269      4.029598e+09 1.403364e+09           2.8714             37.082938       48.056693
      Rabi         66160      2.051688e+09 9.479091e+08           2.1644             18.880943       32.460130
    Winter          6050      4.345498e+08 2.195979e+08           1.9788              3.999006        7.519893
    Summer         14810      1.706344e+08 7.531401e+07           2.2656              1.570287        2.579047
    Autumn          4924      6.439408e+07 4.806907e+07           1.3396              0.592596        1.646074


## 4. Yearly Record Distribution
Validates observation density per harvest year.


In [5]:
record_dist_df = get_yearly_record_distribution()
print(record_dist_df.to_string(index=False))


 year  record_count  states_reporting  districts_reporting  crops_reported
 1997          8860                23                  474              57
 1998         11259                27                  497              59
 1999         12441                28                  515              59
 2000         13553                31                  546              61
 2001         13293                31                  545              61
 2002         16535                32                  568             106
 2003         17139                32                  583             103
 2004         13858                33                  588              53
 2005         13519                33                  593              53
 2006         13976                32                  583              53
 2007         14269                32                  606              54
 2008         14225                30                  582              54
 2009         13767      